In [2]:
from pulp import *


In [5]:
lp = LpProblem("toys_problem",LpMaximize)

x1 = LpVariable(name="soldier", lowBound=0, cat="integer")
x2 = LpVariable(name="train", lowBound=0, cat="integer")

lp += 3*x1 + 2*x2

lp += 2*x1 + x2 <= 100
lp += x1 + x2 <= 80
lp += x1 <= 40

status = lp.solve()

for var in lp.variables():
    print(var.name, var.value())
print(lp.objective.value())

soldier 20.0
train 60.0
180.0


In [7]:
lp = LpProblem("diet_problem",LpMinimize)

x1 = LpVariable(name="meat", lowBound=0, cat="continuous")
x2 = LpVariable(name="rice", lowBound=0, cat="continuous")
x3 = LpVariable(name="beans", lowBound=0, cat="continuous")
x4 = LpVariable(name="sugar", lowBound=0, cat="continuous")
x5 = LpVariable(name="lettace", lowBound=0, cat="continuous")
x6 = LpVariable(name="orange", lowBound=0, cat="continuous")

lp += 0.5*x1 + 0.18*x2 + 0.2*x3 + 0.16*x4 + 0.30*x5 + 0.1*x6

lp += 2.25*x1 + 3.64*x2 + 3.37*x3 + 3.85*x4 + 0.15*x5 + 0.42*x6 >= 3200
lp += 0.01*(7*x1 + 2*x3 + 87*x5 + 13*x6) >= 750
lp += 0.01*(3*x3 + 12*x5 + 59*x6) >= 70
lp += 2.9*x1 + 1.3*x2 + 7.6*x3 + 0.1*x4 + 1.3*x5 + 0.7*x6 >= 10
lp += 0.01*(11*x1 + 9*x2 + 86*x3 +  43*x5 + 34*x6) >= 650

status = lp.solve()

for var in lp.variables():
    print(var.name, var.value())
print(lp.objective.value())

beans 328.55598
lettace 854.51595
meat 0.0
orange 0.0
rice 0.0
sugar 510.28285
403.711237


In [3]:
x = [1,2,3,4,2]
print(max(x))

4


In [1]:
k = []
k.append(1)
print(k)
k.append(2)
print(k)

[1]
[1, 2]


In [ ]:
import numpy as np
x = 0
for i in range(1,1000000):
    x += i**(-2)

print(x)
print(np.pi**2/6)

1.6449339668472496
1.6449340668482264


In [13]:
x = 0
for i in range(1,1000000):
    x += i**(-2)

print(x)

1.64493306684777


In [14]:
10):
    x += i**(-2)

print(x)
print(np.pi**2/6)

SyntaxError: unmatched ')' (855883477.py, line 1)

In [1]:
import numpy as np

def para_forma_padrao(c, A, b):
    """
    Converte um problema de programação linear para a forma padrão (tableau).
    Assume um problema de maximização com restrições <=.

    Args:
        c (np.array): Coeficientes da função objetivo.
        A (np.array): Matriz de coeficientes das restrições.
        b (np.array): Vetor de termos independentes das restrições.

    Returns:
        np.array: O tableau inicial do Simplex.
    """
    num_restricoes, num_variaveis = A.shape

    # Adiciona variáveis de folga (uma para cada restrição)
    A_padrao = np.hstack([A, np.eye(num_restricoes)])

    # Coeficientes da função objetivo na forma padrão (com zeros para as vars de folga)
    c_padrao = np.concatenate([c, np.zeros(num_restricoes)])

    # Monta a primeira linha do tableau (c_N)
    linha_objetivo = np.concatenate([[-c_padrao], [0]])

    # Monta o corpo do tableau
    corpo = np.hstack([A_padrao, b.reshape(-1, 1)])

    # Junta tudo para formar o tableau inicial
    tableau = np.vstack([corpo, linha_objetivo])

    return tableau

def pode_melhorar(tableau):
    """Passo 2: Verifica se a solução atual é ótima.
       A solução é ótima se não houver coeficientes positivos na linha do objetivo."""
    linha_objetivo = tableau[-1, :-1]
    return any(x > 1e-6 for x in linha_objetivo) # Usa tolerância para pontos flutuantes

def encontrar_coluna_pivo(tableau):
    """Passo 3: Encontra a coluna pivô (variável a entrar na base).
       É a coluna com o maior valor positivo na linha do objetivo."""
    linha_objetivo = tableau[-1, :-1]
    return np.argmax(linha_objetivo)

def encontrar_linha_pivo(tableau, coluna_pivo):
    """Passo 4 e 5: Encontra a linha pivô (variável a sair da base)
       pelo teste da razão mínima."""
    b = tableau[:-1, -1]
    A_k = tableau[:-1, coluna_pivo]

    # Passo 4: Verifica se o problema é ilimitado
    if all(val <= 1e-6 for val in A_k):
        raise ValueError("O problema de programação linear é ilimitado.")

    razoes = []
    for i in range(len(b)):
        if A_k[i] > 1e-6: # A_ik > 0
            razoes.append(b[i] / A_k[i])
        else:
            razoes.append(float('inf')) # Razão infinita para não ser selecionada

    return np.argmin(razoes)

def pivotar(tableau, linha_pivo, coluna_pivo):
    """Passo 1 (próxima iteração), 6 e 7: Realiza a operação de pivoteamento
       para atualizar a base."""
    num_linhas, _ = tableau.shape
    novo_tableau = np.copy(tableau)

    # 1. Normaliza a linha pivô (divide pelo elemento pivô)
    elemento_pivo = novo_tableau[linha_pivo, coluna_pivo]
    novo_tableau[linha_pivo, :] /= elemento_pivo

    # 2. Zera os outros elementos da coluna pivô
    for i in range(num_linhas):
        if i != linha_pivo:
            fator = novo_tableau[i, coluna_pivo]
            novo_tableau[i, :] -= fator * novo_tableau[linha_pivo, :]

    return novo_tableau

def extrair_solucao(tableau, num_variaveis_originais):
    """Extrai a solução final do tableau."""
    num_linhas, _ = tableau.shape
    solucao = np.zeros(num_variaveis_originais)
    
    # Encontra as variáveis básicas e seus valores
    for j in range(num_variaveis_originais):
        coluna = tableau[:-1, j]
        # Se a coluna é um vetor da base canônica (um '1' e o resto '0')
        if sum(coluna) == 1 and len(coluna[coluna == 1]) == 1:
            linha = np.where(coluna == 1)[0][0]
            solucao[j] = tableau[linha, -1]

    valor_otimo = -tableau[-1, -1]
    return solucao, valor_otimo

def simplex(c, A, b):
    """
    Função principal que executa o algoritmo Simplex.
    """
    num_variaveis_originais = A.shape[1]
    tableau = para_forma_padrao(c, A, b)
    print("Tableau Inicial:")
    print(tableau)
    print("-" * 30)

    iteracao = 1
    while pode_melhorar(tableau):
        print(f"Iteração {iteracao}:")
        coluna_pivo = encontrar_coluna_pivo(tableau)
        print(f"Variável a entrar na base (coluna pivô): x{coluna_pivo + 1}")
        
        try:
            linha_pivo = encontrar_linha_pivo(tableau, coluna_pivo)
            print(f"Variável a sair da base (linha pivô): {linha_pivo + 1}")
        except ValueError as e:
            print(e)
            return None, None
        
        tableau = pivotar(tableau, linha_pivo, coluna_pivo)
        print("Novo Tableau:")
        print(tableau)
        print("-" * 30)
        iteracao += 1

    solucao, valor_otimo = extrair_solucao(tableau, num_variaveis_originais)
    
    return solucao, valor_otimo

# --- Exemplo de Uso ---
if __name__ == '__main__':
    # Problema Exemplo:
    # Maximizar Z = 3x1 + 5x2
    # Sujeito a:
    # x1 <= 4
    # 2x2 <= 12
    # 3x1 + 2x2 <= 18
    # x1, x2 >= 0

    # Coeficientes da função objetivo
    c = np.array([3, 5])
    
    # Matriz de coeficientes das restrições
    A = np.array([
        [1, 0],
        [0, 2],
        [3, 2]
    ])
    
    # Vetor de termos independentes
    b = np.array([4, 12, 18])

    try:
        solucao, valor_otimo = simplex(c, A, b)
        if solucao is not None:
            print("\nSolução Ótima Encontrada:")
            for i, val in enumerate(solucao):
                print(f"x{i+1} = {val:.2f}")
            print(f"Valor Ótimo (Z) = {valor_otimo:.2f}")
    except Exception as e:
        print(f"Ocorreu um erro: {e}")

Ocorreu um erro: all the input arrays must have same number of dimensions, but the array at index 0 has 2 dimension(s) and the array at index 1 has 1 dimension(s)


In [2]:
import numpy as np

def para_forma_padrao(c, A, b):
    """
    Converte um problema de programação linear para a forma padrão (tableau).
    Assume um problema de maximização com restrições <=.

    Args:
        c (np.array): Coeficientes da função objetivo.
        A (np.array): Matriz de coeficientes das restrições.
        b (np.array): Vetor de termos independentes das restrições.

    Returns:
        np.array: O tableau inicial do Simplex.
    """
    num_restricoes, num_variaveis = A.shape

    # Adiciona variáveis de folga (uma para cada restrição)
    A_padrao = np.hstack((A, np.eye(num_restricoes)))

    # Coeficientes da função objetivo na forma padrão (com zeros para as vars de folga)
    c_padrao = np.concatenate((c, np.zeros(num_restricoes)))

    # Monta a última linha do tableau (linha do objetivo)
    # CORREÇÃO APLICADA AQUI: removidos colchetes extras
    linha_objetivo = np.concatenate((-c_padrao, [0]))

    # Monta o corpo do tableau
    corpo = np.hstack((A_padrao, b.reshape(-1, 1)))

    # Junta tudo para formar o tableau inicial
    tableau = np.vstack((corpo, linha_objetivo))

    return tableau

def pode_melhorar(tableau):
    """Passo 2: Verifica se a solução atual é ótima.
       A solução é ótima se não houver coeficientes positivos na linha do objetivo."""
    linha_objetivo = tableau[-1, :-1]
    return any(x > 1e-6 for x in linha_objetivo) # Usa tolerância para pontos flutuantes

def encontrar_coluna_pivo(tableau):
    """Passo 3: Encontra a coluna pivô (variável a entrar na base).
       É a coluna com o maior valor positivo na linha do objetivo."""
    linha_objetivo = tableau[-1, :-1]
    return np.argmax(linha_objetivo)

def encontrar_linha_pivo(tableau, coluna_pivo):
    """Passo 4 e 5: Encontra a linha pivô (variável a sair da base)
       pelo teste da razão mínima."""
    b = tableau[:-1, -1]
    A_k = tableau[:-1, coluna_pivo]

    # Passo 4: Verifica se o problema é ilimitado
    if all(val <= 1e-6 for val in A_k):
        raise ValueError("O problema de programação linear é ilimitado.")

    razoes = []
    for i in range(len(b)):
        if A_k[i] > 1e-6: # A_ik > 0
            razoes.append(b[i] / A_k[i])
        else:
            razoes.append(float('inf')) # Razão infinita para não ser selecionada

    return np.argmin(razoes)

def pivotar(tableau, linha_pivo, coluna_pivo):
    """Passo 1 (próxima iteração), 6 e 7: Realiza a operação de pivoteamento
       para atualizar a base."""
    num_linhas, _ = tableau.shape
    novo_tableau = np.copy(tableau)

    # 1. Normaliza a linha pivô (divide pelo elemento pivô)
    elemento_pivo = novo_tableau[linha_pivo, coluna_pivo]
    novo_tableau[linha_pivo, :] /= elemento_pivo

    # 2. Zera os outros elementos da coluna pivô
    for i in range(num_linhas):
        if i != linha_pivo:
            fator = novo_tableau[i, coluna_pivo]
            novo_tableau[i, :] -= fator * novo_tableau[linha_pivo, :]

    return novo_tableau

def extrair_solucao(tableau, num_variaveis_originais):
    """Extrai a solução final do tableau."""
    solucao = np.zeros(num_variaveis_originais)
    
    # Encontra as variáveis básicas e seus valores
    for j in range(num_variaveis_originais):
        coluna = tableau[:-1, j]
        # Se a coluna é um vetor da base canônica (um '1' e o resto '0')
        soma_elementos = np.sum(coluna)
        num_zeros = len(coluna) - np.count_nonzero(coluna)
        
        if soma_elementos == 1 and num_zeros == len(coluna) - 1:
            linha = np.where(coluna == 1)[0][0]
            solucao[j] = tableau[linha, -1]

    valor_otimo = -tableau[-1, -1]
    return solucao, valor_otimo

def simplex(c, A, b):
    """
    Função principal que executa o algoritmo Simplex.
    """
    num_variaveis_originais = A.shape[1]
    tableau = para_forma_padrao(c, A, b)
    print("Tableau Inicial:")
    print(np.round(tableau, 2))
    print("-" * 40)

    iteracao = 1
    while pode_melhorar(tableau):
        print(f"Iteração {iteracao}:")
        coluna_pivo = encontrar_coluna_pivo(tableau)
        print(f"Variável a entrar na base (coluna pivô): x{coluna_pivo + 1}")
        
        try:
            linha_pivo = encontrar_linha_pivo(tableau, coluna_pivo)
            print(f"Variável a sair da base (linha pivô): {linha_pivo + 1}")
        except ValueError as e:
            print(e)
            return None, None
        
        tableau = pivotar(tableau, linha_pivo, coluna_pivo)
        print("Novo Tableau:")
        print(np.round(tableau, 2))
        print("-" * 40)
        iteracao += 1

    solucao, valor_otimo = extrair_solucao(tableau, num_variaveis_originais)
    
    return solucao, valor_otimo

# --- Exemplo de Uso ---
if __name__ == '__main__':
    # Problema Exemplo:
    # Maximizar Z = 3x1 + 5x2
    # Sujeito a:
    # x1 <= 4
    # 2x2 <= 12
    # 3x1 + 2x2 <= 18
    # x1, x2 >= 0

    # Coeficientes da função objetivo
    c = np.array([3, 5])
    
    # Matriz de coeficientes das restrições
    A = np.array([
        [1, 0],
        [0, 2],
        [3, 2]
    ])
    
    # Vetor de termos independentes
    b = np.array([4, 12, 18])

    try:
        solucao, valor_otimo = simplex(c, A, b)
        if solucao is not None:
            print("\n✅ Solução Ótima Encontrada:")
            for i, val in enumerate(solucao):
                print(f"  x{i+1} = {val:.2f}")
            print(f"  Valor Ótimo (Z) = {valor_otimo:.2f}")
    except Exception as e:
        print(f"❌ Ocorreu um erro: {e}")
# Importa a função simplex do arquivo que criamos anteriormente
from simplex_solver import simplex

def resolver_problema_logistica():
    """
    Esta função formula e resolve o problema de transporte
    usando o solver Simplex.
    """
    # --- 1. Dados do Problema (Exemplo: 3 fábricas e 4 lojas) ---
    m = 3  # Número de fábricas
    n = 4  # Número de lojas

    # Capacidade de produção de cada fábrica
    u = np.array([100, 120, 80])  # Total de produção: 300

    # Demanda de cada loja
    l = np.array([60, 50, 70, 40])  # Total de demanda: 220

    # Custos de transporte: fábrica -> hub
    a = np.array([2, 5, 3])  # Custos para o Hub A
    b = np.array([4, 1, 6])  # Custos para o Hub B

    # Custos de transporte: hub -> loja
    d = np.array([3, 5, 4, 2])  # Custos do Hub A
    e = np.array([6, 2, 5, 3])  # Custos do Hub B

    # Verifica se a produção total é suficiente para a demanda
    if np.sum(u) < np.sum(l):
        print("Produção insuficiente para atender à demanda. O problema não tem solução.")
        return

    # --- 2. Mapeamento de Variáveis para um Vetor ---
    # O vetor de decisão 'x' terá a seguinte ordem:
    # x_1A, ..., x_mA  (m variáveis)
    # x_1B, ..., x_mB  (m variáveis)
    # y_A1, ..., y_An  (n variáveis)
    # y_B1, ..., y_Bn  (n variáveis)
    num_vars = 2 * m + 2 * n

    # --- 3. Construção da Função Objetivo (Maximizar -Z) ---
    # Lembre-se: min(Z) = -max(-Z), então negamos todos os custos
    c = -np.concatenate([a, b, d, e])

    # --- 4. Construção das Matrizes de Restrição (A e b) ---
    A_restricoes = []
    b_restricoes = []

    # Restrição 1: Capacidade da Fábrica (m restrições do tipo <=)
    for i in range(m):
        linha = np.zeros(num_vars)
        linha[i] = 1        # Coeficiente para x_iA
        linha[i + m] = 1    # Coeficiente para x_iB
        A_restricoes.append(linha)
        b_restricoes.append(u[i])

    # Restrição 2: Demanda da Loja (n restrições de igualdade -> 2n restrições <=)
    for j in range(n):
        # Parte 1: y_Aj + y_Bj <= l_j
        linha1 = np.zeros(num_vars)
        linha1[2 * m + j] = 1       # Coeficiente para y_Aj
        linha1[2 * m + n + j] = 1   # Coeficiente para y_Bj
        A_restricoes.append(linha1)
        b_restricoes.append(l[j])

        # Parte 2: -y_Aj - y_Bj <= -l_j
        linha2 = -linha1
        A_restricoes.append(linha2)
        b_restricoes.append(-l[j])

    # Restrição 3: Fluxo nos Hubs (2 restrições de igualdade -> 4 restrições <=)
    # Hub A: sum(x_iA) - sum(y_Aj) = 0
    linha_hub_a = np.zeros(num_vars)
    linha_hub_a[0:m] = 1             # Coefs para x_iA
    linha_hub_a[2 * m : 2 * m + n] = -1 # Coefs para y_Aj
    A_restricoes.append(linha_hub_a)   # <= 0
    b_restricoes.append(0)
    A_restricoes.append(-linha_hub_a)  # <= 0
    b_restricoes.append(0)

    # Hub B: sum(x_iB) - sum(y_Bj) = 0
    linha_hub_b = np.zeros(num_vars)
    linha_hub_b[m : 2 * m] = 1               # Coefs para x_iB
    linha_hub_b[2 * m + n : 2 * m + 2 * n] = -1 # Coefs para y_Bj
    A_restricoes.append(linha_hub_b)   # <= 0
    b_restricoes.append(0)
    A_restricoes.append(-linha_hub_b)  # <= 0
    b_restricoes.append(0)
    
    A = np.array(A_restricoes)
    b = np.array(b_restricoes)
    
    # --- 5. Resolução do Problema ---
    print("Iniciando a resolução do problema de logística...")
    # Ocultamos a saída detalhada do Simplex para focar no resultado final
    solucao_vars, valor_max_neg_z = simplex(c, A, b)

    # --- 6. Apresentação dos Resultados ---
    if solucao_vars is not None:
        custo_minimo = -valor_max_neg_z  # Corrigindo o valor para o problema de minimização

        print("\n" + "="*50)
        print("✅ Solução Ótima Encontrada!")
        print(f"  Custo Mínimo Total de Transporte: {custo_minimo:.2f}")
        print("="*50)

        print("\n🚚 Rota Fábrica -> Hub:")
        x_iA = solucao_vars[0:m]
        x_iB = solucao_vars[m:2*m]
        for i in range(m):
            if x_iA[i] > 1e-6: print(f"  Fábrica {i+1} -> Hub A: {x_iA[i]:.2f} unidades")
            if x_iB[i] > 1e-6: print(f"  Fábrica {i+1} -> Hub B: {x_iB[i]:.2f} unidades")

        print("\n📦 Rota Hub -> Loja:")
        y_Aj = solucao_vars[2*m : 2*m+n]
        y_Bj = solucao_vars[2*m+n : 2*m+2*n]
        for j in range(n):
            if y_Aj[j] > 1e-6: print(f"  Hub A -> Loja {j+1}: {y_Aj[j]:.2f} unidades")
            if y_Bj[j] > 1e-6: print(f"  Hub B -> Loja {j+1}: {y_Bj[j]:.2f} unidades")
            
    else:
        print("\nNão foi possível encontrar uma solução para o problema.")

if __name__ == '__main__':
    # Renomeia o arquivo do solver se necessário
    import sys
    try:
        __import__("simplex_solver")
    except ImportError:
        print("ERRO: Certifique-se de que o código Simplex está salvo como 'simplex_solver.py' no mesmo diretório.")
        sys.exit(1)
        
    resolver_problema_logistica()

Tableau Inicial:
[[ 1.  0.  1.  0.  0.  4.]
 [ 0.  2.  0.  1.  0. 12.]
 [ 3.  2.  0.  0.  1. 18.]
 [-3. -5. -0. -0. -0.  0.]]
----------------------------------------

✅ Solução Ótima Encontrada:
  x1 = 0.00
  x2 = 0.00
  Valor Ótimo (Z) = -0.00


ModuleNotFoundError: No module named 'simplex_solver'